In [ ]:
import sys
import os
import pickle
from tqdm.auto import tqdm

BASE_DIR = os.getcwd()
BASE_DIR

In [ ]:
# Set the primary font for visualization
from matplotlib import font_manager as fm

# Define the font path and initialize FontProperties
font_path = BASE_DIR + '/NotoSans-VariableFont_wdth,wght.ttf'
prop = fm.FontProperties(fname=font_path)

# Step 2: Inter-event time

## Step 2-1: Overall IET (Consecutive edits)

In [ ]:
import os
import pickle
from collections import Counter
from tqdm.auto import tqdm
import ciso8601

def Get_editor_overall_IET(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Calculates the Inter-Event Time (IET) for all editors within the conditioned history dataset.
    Quantifies the time intervals between consecutive edits to analyze temporal activity patterns.
    """
    
    # DATA_DIR: Source directory for conditioned history logs
    # TARGET_DIR: Destination directory for IET distribution and statistics
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    def load_files():
        """
        Retrieves the serialized editor history dataset.
        """
        history_file_name = f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle'
        history_file_path = os.path.join(DATA_DIR, history_file_name)
        with open(history_file_path, 'rb') as fr:
            editor_histories = pickle.load(fr)
        return editor_histories

    # Initialize data structures for temporal aggregation
    editor_histories = load_files()
    IETs = Counter() # Frequency repository for time durations (in seconds)
    editors = list(editor_histories.keys())

    # Local assignment for high-performance datetime parsing
    parse_datetime = ciso8601.parse_datetime
    # Stage 1: Iterative IET Calculation
    for editor in tqdm(editors, desc="Calculating Overall IETs"):
        histories = editor_histories[editor]
        parsed_history = []
        
        # Convert raw timestamps into datetime objects for arithmetic operations
        for event in histories:
            parsed_history.append((parse_datetime(event[0]), event[1]))
        
        # Ensure chronological ordering to maintain temporal integrity
        parsed_history.sort(key=lambda x: x[0])
        
        # Calculate intervals only for editors with multiple revision events
        len_histories = len(parsed_history)
        if len_histories < 2:
            continue
            
        for nhistory in range(1, len_histories):
            t1, _ = parsed_history[nhistory-1]
            t2, _ = parsed_history[nhistory]
            
            # Record the duration between consecutive events in seconds
            deltat = (t2 - t1).total_seconds()
            IETs[deltat] += 1

    # Stage 2: Data Serialization
    iet_output_file_path = os.path.join(TARGET_DIR, f'editor_overall_IETs_{page_edit_number}_{editor_edit_number}.pickle')
    with open(iet_output_file_path, 'wb') as fw:
        pickle.dump(dict(IETs), fw)

    # Stage 3: Statistical Reporting
    def get_stats_data(iet_dict, label):
        """
        Generates a descriptive statistical summary from the IET frequency distribution.
        """
        if not iet_dict:
            return f"--- {label} Transition Statistics ---\nNo Data Found.\n"
        
        # Sort by duration to compute cumulative metrics
        sorted_items = sorted(iet_dict.items())
        
        total_transitions = sum(count for _, count in sorted_items)
        total_sum = sum(sec * count for sec, count in sorted_items)
        mean_iet = total_sum / total_transitions
        
        # Median Calculation: Identifying the duration at the 50th percentile of total transitions
        median_idx = (total_transitions + 1) / 2
        cum_count = 0
        median_iet_val = None
        for sec, count in sorted_items:
            cum_count += count
            if cum_count >= median_idx:
                median_iet_val = sec
                break

        return (
            f"--- {label} Transition Statistics ---\n"
            f"Total Transitions : {total_transitions:,}\n"
            f"Mean IET          : {mean_iet:.4f} s\n"
            f"Median IET        : {median_iet_val:.4f} s\n"
            f"Min/Max IET       : {sorted_items[0][0]} / {sorted_items[-1][0]}\n"
        )
    
    # Generate and export the statistical summary report
    stats_text = get_stats_data(IETs, "Overall IET")
    stats_report_file_path = os.path.join(TARGET_DIR, f"statistics_overall_IETs_{page_edit_number}_{editor_edit_number}.txt")
    
    with open(stats_report_file_path, "w", encoding="utf-8") as f:
        f.write(stats_text)
        
    print(f"[FINISH] Temporal analysis complete. Saved to: {iet_output_file_path}")
    return

## Step 2-2: Immediate IET (Immediate transition)

In [ ]:
import os
import pickle
import random
from collections import Counter
from tqdm.auto import tqdm
import ciso8601

def Get_editor_immediate_IET(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Calculates the Inter-Event Time (IET) for immediate transitions, categorized by 
    structural hyperlink properties (directed, reciprocal, or non-linked).
    """
    # Set seed for reproducibility of non-hyperlink sampling
    random.seed(20251103)
    
    # DATA_DIR: Source directory for interaction pairs and histories
    # TARGET_DIR: Destination for IET distributions and statistical reports
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    def load_files():
        """
        Retrieves serialized interaction pairs and editor activity logs.
        """
        pair_file_path = os.path.join(DATA_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        hist_file_path = os.path.join(DATA_DIR, f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        
        with open(pair_file_path, 'rb') as fr: pair_list = pickle.load(fr)
        with open(hist_file_path, 'rb') as fr: editor_histories = pickle.load(fr)
        return pair_list, editor_histories

    # Stage 1: Network Topology Preparation
    pair_list, editor_histories = load_files()
    
    # hyperlink_set: Stores unique undirected structural edges
    hyperlink_set = set()
    for p1, p2 in pair_list:
        hyperlink_set.add(tuple(sorted((p1, p2))))
        
    # link_set: Stores directed structural edges for orientation analysis
    link_set = set(tuple(p) for p in pair_list)
    del pair_list 
    
    # link_im_IETs(hyperlink), full_nonlink_im_IETs(non-hyperlink), sample_nonlink_IETs(non-hyperlink)
    link_im_IETs, full_nonlink_im_IETs, sample_nonlink_im_IETs = Counter(), Counter(), Counter()
    # forward_im_IETs(only p1 -> p2), backward_im_IETs(only p2 -> p1), reciproacl_im_IETs(p1 <-> p2)
    forward_im_IETs, backward_im_IETs, reciprocal_im_IETs = Counter(), Counter(), Counter()
    
    link_im_pairs, full_nonlink_im_pairs, sample_nonlink_im_pairs = set(), set(), set()
    forward_im_pairs, backward_im_pairs, reciprocal_im_pairs = set(), set(), set()
    
    editors = list(editor_histories.keys())
    parse_datetime = ciso8601.parse_datetime 
    
    # Stage 2: Temporal Transition Analysis
    for editor in tqdm(editors):
        histories = editor_histories[editor]
        if len(histories) < 2: continue
        histories.sort(key=lambda x: x[0])

        page_list = list({e[1] for e in histories})
        if len(page_list) < 2:
            continue
        
        editor_full_nonlink = set()
        editor_link = set()

        # Identify immediate transitions and calculate deltat
        for i in range(1, len(histories)):
            t1_str, p1 = histories[i-1]
            t2_str, p2 = histories[i]
            if p1 == p2: continue

            deltat = (parse_datetime(t2_str) - parse_datetime(t1_str)).total_seconds()
            
            has_fwd = (p1, p2) in link_set
            has_bwd = (p2, p1) in link_set
            pair_undirected = tuple(sorted((p1, p2))) # 방향 무시용

            # Categorize based on structural link properties
            if has_fwd and has_bwd:
                link_im_IETs[deltat] += 1
                reciprocal_im_IETs[deltat] += 1
                link_im_pairs.add(pair_undirected)
                reciprocal_im_pairs.add(pair_undirected)
                editor_link.add(pair_undirected)
            elif has_fwd:
                link_im_IETs[deltat] += 1
                forward_im_IETs[deltat] += 1
                link_im_pairs.add((p1, p2))
                forward_im_pairs.add((p1, p2))
                editor_link.add((p1, p2))
            elif has_bwd:
                link_im_IETs[deltat] += 1
                backward_im_IETs[deltat] += 1
                link_im_pairs.add((p2, p1))
                backward_im_pairs.add((p2, p1))
                editor_link.add((p2, p1))
            else:
                full_nonlink_im_IETs[deltat] += 1
                full_nonlink_im_pairs.add(pair_undirected)
                editor_full_nonlink.add(pair_undirected)
    
        # Non-hyperlink sampling to control for volume bias
        sample_random_pairs = set()
        actual_target = min(len(editor_link), len(editor_full_nonlink))
        editor_full_nonlink = list(editor_full_nonlink)
        attempts = 0
        while len(sample_random_pairs) < actual_target:
            attempts += 1
            if attempts > actual_target * 100:
                break
            pair = random.choice(editor_full_nonlink)
            if pair not in sample_random_pairs:
                sample_random_pairs.add(pair)
        
        # Recalculate IETs for sampled non-hyperlink transitions
        for i in range(1, len(histories)):
            t1_str, p1 = histories[i-1]
            t2_str, p2 = histories[i]
            if p1 == p2: continue

            deltat = (parse_datetime(t2_str) - parse_datetime(t1_str)).total_seconds()
            
            has_nonlink_fwd = (p1, p2) in sample_random_pairs
            has_nonlink_bwd = (p2, p1) in sample_random_pairs
            
            if (p1, p2) in sample_random_pairs:
                sample_nonlink_im_IETs[deltat] += 1
                sample_nonlink_im_pairs.add((p1, p2))
            elif (p2, p1) in sample_random_pairs:
                sample_nonlink_im_IETs[deltat] += 1
                sample_nonlink_im_pairs.add((p2, p1))

    # Stage 3: Results Serialization
    IETs_outputs = {'hyperlink': link_im_IETs, 'full_nonhyperlink': full_nonlink_im_IETs, 
                    'sample_nonhyperlink': sample_nonlink_im_IETs, 'reciprocal': reciprocal_im_IETs, 
                    'forward': forward_im_IETs, 'backward': backward_im_IETs}
    pairs_outputs = {'hyperlink': link_im_pairs, 'full_nonhyperlink': full_nonlink_im_pairs, 
                    'sample_nonhyperlink': sample_nonlink_im_pairs, 'reciprocal': reciprocal_im_pairs, 
                    'forward': forward_im_pairs, 'backward': backward_im_pairs}
    
    
    iet_output_file_path = os.path.join(TARGET_DIR, f"editor_immediate_IETs_{page_edit_number}_{editor_edit_number}_combined.pickle")
    pairs_output_file_path = os.path.join(TARGET_DIR, f"editor_immediate_pairs_{page_edit_number}_{editor_edit_number}_combined.pickle")
    
    with open(iet_output_file_path, 'wb') as fw: pickle.dump(IETs_outputs, fw)
    with open(pairs_output_file_path, 'wb') as fw: pickle.dump(pairs_outputs, fw)
    
    
    # Stage 4: Comprehensive Statistical Reporting
    def get_im_stats_data(iet_dict, pairs, label):
        if not iet_dict:
            return f"--- {label} Transition Statistics ---\nNo Data Found.\n"
        
        sorted_items = sorted(iet_dict.items())
        
        total_transitions = sum(count for _, count in sorted_items)
        total_sum = sum(sec * count for sec, count in sorted_items)
        mean_iet = total_sum / total_transitions
        
        # 중앙값(Median) 계산
        median_idx = (total_transitions + 1) / 2
        cum_count = 0
        median_iet_val = None
        for sec, count in sorted_items:
            cum_count += count
            if cum_count >= median_idx:
                median_iet_val = sec
                break

        return (
            f"--- {label} Transition Statistics ---\n"
            f"Total Transitions : {total_transitions:,}\n"
            f"Unique pairs : {len(pairs):,}\n"
            f"Density       : {total_transitions/len(pairs):.4f} s\n"
            f"Mean IET          : {mean_iet:.4f} s\n"
            f"Median IET        : {median_iet_val:.4f} s\n"
            f"Min/Max IET       : {sorted_items[0][0]} / {sorted_items[-1][0]}\n"
        )

    final_report = ""
    for category in IETs_repos.keys():
        final_report += "\n" + get_im_stats_data(IETs_repos[category], pairs_repos[category], f"Immediate IET ({category})")
        
    stats_report_file_path = os.path.join(TARGET_DIR, f"statistics_immediate_IETs_{page_edit_number}_{editor_edit_number}.txt")
    with open(stats_report_file_path, "w", encoding="utf-8") as f:
        f.write(final_report)
    
    print(f"[FINISH] Immediate IET analysis consolidated: {iet_output_file_path}")
    return

## Step 2-3: Long-term IET (Long-term transition)

### Step 2-3-1: Hyperlinked pair

In [ ]:
import os
import pickle
import math
import ciso8601
from collections import Counter, defaultdict
from tqdm.auto import tqdm

def Get_editor_longterm_IET_hyperlink(BASE_DIR, page_edit_number, editor_edit_number, part):
    """
    Analyzes long-term recurrence intervals between hyperlinked pages within an editor's full history.
    Categorizes intervals based on the structural orientation of the hyperlink network.
    """
    
    # DATA_DIR: Source directory for history logs and interaction pairs
    # TARGET_DIR: Destination for serialized long-term IET distributions
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    def load_files():
        """
        Retrieves serialized interaction pairs and filtered editor histories.
        """
        pair_file_path = os.path.join(DATA_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        hist_file_path = os.path.join(DATA_DIR, f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        
        with open(pair_file_path, 'rb') as fr: pair_list = pickle.load(fr)
        with open(hist_file_path, 'rb') as fr: editor_histories = pickle.load(fr)
        return pair_list, editor_histories

    pair_list, editor_histories = load_files()
    print('[INFO] Input files loaded successfully.')
        
    # Partitioning logic for parallel/segment-based processing
    num_parts = 40
    if part == 1: start_index, end_index = math.floor(total_length * 0), math.floor(total_length * (1/40))
    elif part == 2: start_index, end_index = math.floor(total_length * (1/40)), math.floor(total_length * (5/40))
    elif part == 3: start_index, end_index = math.floor(total_length * (5/40)), math.floor(total_length * (10/40))
    elif part == 4: start_index, end_index = math.floor(total_length * (10/40)), total_length
    part_range = [start_index, end_index]

    
    # Stage 1: Build Global Adjacency Lookup
    adj = defaultdict(set)
    for p1, p2 in pair_list:
        adj[p1].add(p2)
    
    parse_datetime = ciso8601.parse_datetime
    editors = list(editor_histories.keys())
    total_length = len(editors)
    
    # Repositories for Long-term IET distributions
    reciprocal_lt_IETs = Counter()
    forward_lt_IETs = Counter()
    backward_lt_IETs = Counter()
    lt_IETs = Counter()
    
    # Track unique pairs associated with long-term intervals
    pair_reciprocal = set()
    pair_fwd = set()
    pair_bwd = set()
    pair_link = set()

    # Stage 2: Segment-based IET Extraction
    for neditor in tqdm(range(part_range[0], part_range[1])):
        editor = editors[neditor]
        # Reconstruct and sort individual history
        history = []
        for t_str, pg in editor_histories[editor]:
            history.append((parse_datetime(t_str), pg))
        
        history.sort(key=lambda x: x[0])
        
        visited_pages = {event[1] for event in history}
        if len(visited_pages) < 2:
            continue
        
        # Map each visited page to its list of occurrences
        page_to_times = defaultdict(list)
        for ts, pg in history:
            page_to_times[pg].append((ts, pg))

        # Identify hyperlinked pairs within the editor's activity subspace
        local_links = []
        for p1 in visited_pages:
            if p1 in adj:
                # Intersect adjacency with visited pages to find active links
                for p2 in (adj[p1] & visited_pages):
                    local_links.append((p1, p2))

        processed_pairs = set()
        for p1, p2 in local_links:
            if p1 == p2: continue
            
            norm_pair = tuple(sorted((p1, p2)))
            if norm_pair in processed_pairs: continue
            
            # Structural property verification
            is_reciprocal = (p2 in adj and p1 in adj[p2])
            occurred_fwd = False
            occurred_bwd = False
            occurred_reci = False
            occurred_link = False
            
            # Combine all timestamps for the page pair and sort chronologically
            combined_ts = sorted(page_to_times[p1] + page_to_times[p2], key=lambda x: x[0])
            
            for i in range(1, len(combined_ts)):
                t1, pg1 = combined_ts[i-1]
                t2, pg2 = combined_ts[i]
                
                # Calculate IET only for transitions between different pages in the pair
                if pg1 != pg2:
                    deltat = (t2 - t1).total_seconds()
                    lt_IETs[deltat] += 1
                    occurred_link = True
                    if is_reciprocal:
                        reciprocal_lt_IETs[deltat] += 1
                        occurred_reci = True
                    else:
                        # Determine alignment with link direction (p1 -> p2)
                        if p2 in adj[p1]:
                            if pg1 == p1: # Chronological order matches structural link
                                forward_lt_IETs[deltat] += 1
                                occurred_fwd = True
                            else: # Chronological order is opposite to link
                                backward_lt_IETs[deltat] += 1
                                occurred_bwd = True
                        else: # Structural link is p2 -> p1
                            if pg1 == p2: 
                                forward_lt_IETs[deltat] += 1
                                occurred_fwd = True
                            else:
                                backward_lt_IETs[deltat] += 1
                                occurred_bwd = True
            processed_pairs.add(norm_pair)

            # Record unique pairs for each recurrence category
            if occurred_link:
                pair_link.add(norm_pair)
            if occurred_reci:
                pair_reciprocal.add(norm_pair)
            if occurred_fwd:
                if p2 in adj[p1]: pair_fwd.add((p1, p2))
                else: pair_fwd.add((p2, p1))
            if occurred_bwd:
                if p2 in adj[p1]: pair_bwd.add((p1, p2))
                else: pair_bwd.add((p2, p1))

    # Stage 3: Results Serialization
    iet_results = {
        'longterm_IETs': dict(lt_IETs), 
        'longterm_reciprocal_IETs': dict(reciprocal_lt_IETs), 
        'longterm_forward_IETs': dict(forward_lt_IETs), 
        'longterm_backward_IETs': dict(backward_lt_IETs)
    }

    iet_file_path = os.path.join(TARGET_DIR, f'editor_longterm_IETs_{page_edit_number}_{editor_edit_number}_hyperlink_part{part}.pickle')
    with open(iet_file_path, 'wb') as fw:
        pickle.dump(iet_results, fw)
                                  
    pair_results = {
        'link': pair_link,
        'reci': pair_reciprocal,
        'fwd': pair_fwd,
        'bwd': pair_bwd
    }
    pair_file_path = os.path.join(TARGET_DIR, f'editor_longterm_pairs_{page_edit_number}_{editor_edit_number}_hyperlink_part{part}.pickle')
    with open(pair_file_path, 'wb') as fw:
        pickle.dump(pair_results, fw)

    print(f"[SUCCESS] Long-term IET analysis for part {part} completed: {iet_file_path}")
    return

In [ ]:
import os
import pickle
from collections import defaultdict

def Merge_longterm_IETs_hyperlink(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Consolidates partitioned long-term IET results into a unified dataset and 
    generates descriptive statistics for each recurrence category.
    """
    
    # DATA_DIR: Source directory for partitioned IET files
    # TARGET_DIR: Destination for merged datasets and statistical reports
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(DATA_DIR, exist_ok=True)
    
    # Initializing repositories for consolidated recurrence metrics
    IETs = defaultdict(int)           # Total structural recurrence
    forward_IETs = defaultdict(int)   # Link-consistent (p1 -> ... -> p2)
    backward_IETs = defaultdict(int)  # Link-inconsistent (p2 -> ... -> p1)
    reciprocal_IETs = defaultdict(int)# Bidirectional (p1 <-> p2)
    uni_IETs = defaultdict(int)       # Direction-aware mapping (positive=fwd, negative=bwd)
    uni_IETs_entire = defaultdict(int)# Expanded direction-aware mapping (includes reciprocal)
    
    # Stage 1: Iterative Consolidation of Result Segments
    for part in range(1, 5):
        part_file_path = os.path.join(DATA_DIR, f'editor_longterm_IETs_{page_edit_number}_{editor_edit_number}_hyperlink_part{part}.pickle')
        
        if not os.path.exists(part_file_path):
            continue

        with open(part_file_path, 'rb') as fr:
            RESULTS = pickle.load(fr)
            
        # Aggregate Total IETs
        iets = RESULTS['longterm_IETs']
        for iet, count in iets.items():
            IETs[iet] += count
        iets = True
        
        # Aggregate Forward IETs and map to positive Uni-IETs
        forward_iets = RESULTS['longterm_forward_IETs']
        for iet, count in forward_iets.items():
            forward_IETs[iet] += count
            uni_IETs_entire[iet] += count
            uni_IETs[iet] += count
        forward_iets = True
        
        # Aggregate Backward IETs and map to negative Uni-IETs (Link-inconsistent indicator)
        backward_iets = RESULTS['longterm_backward_IETs']
        for iet, count in backward_iets.items():
            backward_IETs[iet] += count
            uni_IETs_entire[-iet] += count # Negative key denotes backward direction
            uni_IETs[-iet] += count
        backward_iets = True
        
        # Aggregate Reciprocal IETs
        reciprocal_iets = RESULTS['longterm_reciprocal_IETs']
        for iet, count in reciprocal_iets.items():
            reciprocal_IETs[iet] += count
            uni_IETs_entire[iet] += count
        reciprocal_iets = True
    
    # Stage 2: Serialize Integrated Datasets
    save_configs = {
        'hyperlink': IETs,
        'uni': uni_IETs,
        'uni_entire': uni_IETs_entire,
        'forward': forward_IETs,
        'backward': backward_IETs,
        'reciprocal': reciprocal_IETs
    }
    
    for suffix, data in save_configs.items():
        integrated_file_path = os.path.join(DATA_DIR, f'editor_longterm_IETs_{page_edit_number}_{editor_edit_number}_{suffix}.pickle')
        with open(integrated_file_path, 'wb') as fw:
            pickle.dump(dict(data), fw)

    # Stage 3: Statistical Reporting
    def calculate_stats(iet_dict, label):
        """
        Computes descriptive statistics (Mean, Median, Min/Max) for a given IET distribution.
        """
        if not iet_dict:
            return f"--- {label} Statistics ---\nNo Data Found.\n\n"
        
        # Sort by duration to compute cumulative distribution metrics
        sorted_items = sorted(iet_dict.items())
        total_transitions = sum(count for _, count in sorted_items)
        total_sum = sum(sec * count for sec, count in sorted_items)
        mean_iet = total_sum / total_transitions
        
        # Median Calculation: Identifying the 50th percentile
        median_idx = (total_transitions + 1) / 2
        cum_count = 0
        median_iet_val = None
        for sec, count in sorted_items:
            cum_count += count
            if cum_count >= median_idx:
                median_iet_val = sec
                break
        
        return (
            f"--- {label} Statistics ---\n"
            f"Total Transitions : {total_transitions:,}\n"
            f"Mean IET          : {mean_iet:.4f} s\n"
            f"Median IET        : {median_iet_val:.4f} s\n"
            f"Min / Max IET     : {sorted_items[0][0]} / {sorted_items[-1][0]}\n\n"
        )

    # Generate and export the combined statistical summary
    all_stats_text = ""
    stat_targets = [
        (IETs, "Total Long-term IET"),
        (forward_IETs, "Forward (Link-consistent)"),
        (backward_IETs, "Backward (Link-inconsistent)"),
        (reciprocal_IETs, "Reciprocal (Bidirectional)")
    ]
    
    for data_dict, label in stat_targets:
        all_stats_text += calculate_stats(data_dict, label)

    stats_report_file_path = os.path.join(DATA_DIR, f"statistics_longterm_IETs_{page_edit_number}_{editor_edit_number}.txt")
    with open(stats_report_file_path, "w", encoding="utf-8") as f:
        f.write(all_stats_text)
        
    print(f"[FINISH] Long-term IET consolidation complete. Report saved to: {stats_report_file_path}")
    return

In [ ]:
import os
import pickle

def Merge_longterm_pairs_hyperlink(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Consolidates partitioned long-term interaction pairs into a unified dataset.
    Aggregates unique structural edges categorized by their recurrence orientation.
    """
    
    # DATA_DIR: Source directory for partitioned long-term pair files
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(DATA_DIR, exist_ok=True)

    # Initializing global sets for each recurrence category to ensure uniqueness
    PAIR_link = set()
    PAIR_reciprocal = set()
    PAIR_fwd = set()
    PAIR_bwd = set()
    
    # Stage 1: Iterative Consolidation of Result Partitions
    for part in range(1, 5):
        # Construct the absolute path for each partitioned pair segment
        part_file_path = os.path.join(DATA_DIR, f'editor_longterm_pairs_{page_edit_number}_{editor_edit_number}_hyperlink_part{part}.pickle')
        
        if not os.path.exists(part_file_path):
            continue

        with open(part_file_path, 'rb') as fr:
            RESULTS = pickle.load(fr)
            
        # Update global sets with the contents of each partition
        PAIR_link.update(RESULTS['link'])
        PAIR_reciprocal.update(RESULTS['reci'])
        PAIR_fwd.update(RESULTS['fwd'])
        PAIR_bwd.update(RESULTS['bwd'])
        
        # Explicitly release localized partition data to optimize memory usage
        del RESULTS
        
    # Stage 2: Data Consolidation and Serialization
    pairs_combine = {
        'hyperlink': PAIR_link, 
        'reciprocal': PAIR_reciprocal, 
        'forward': PAIR_fwd, 
        'backward': PAIR_bwd
    }
    
    # Define the standardized output path for the finalized integrated pairs dataset
    integrated_pairs_file_path = os.path.join(DATA_DIR, f'editor_longterm_pairs_{page_edit_number}_{editor_edit_number}_hyperlink.pickle')
    
    with open(integrated_pairs_file_path, 'wb') as fw:
        pickle.dump(pairs_combine, fw)

    print(f"[FINISH] Long-term pair consolidation complete. Saved to: {integrated_pairs_file_path}")
    return

### Step 1-3-2: Non-hyperlinked pair

In [ ]:
import os
import pickle
import random
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import ciso8601
from itertools import combinations

def Get_editor_longterm_IET_nonhyperlink(BASE_DIR, page_edit_number, editor_edit_number, part):
    """
    Analyzes recurrence intervals between non-hyperlinked page pairs within an editor's full history.
    Uses randomized sampling to create a control group for structural connectivity analysis.
    """
    # Set seed for reproducibility of randomized pair sampling
    random.seed(20251103)
    
    # DATA_DIR: Source directory for history logs and interaction pairs
    # TARGET_DIR: Destination for serialized non-hyperlink IET distributions
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    def load_files():
        """
        Retrieves serialized interaction pairs and filtered editor histories.
        """
        pair_file_path = os.path.join(DATA_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        hist_file_path = os.path.join(DATA_DIR, f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        
        with open(pair_file_path, 'rb') as fr: pair_list = pickle.load(fr)
        with open(hist_file_path, 'rb') as fr: editor_histories = pickle.load(fr)
        return pair_list, editor_histories

    pair_list, editor_histories = load_files()
    print('[INFO] Input files loaded for non-hyperlink analysis.')
    
    parse_datetime = ciso8601.parse_datetime
    
    # Construct a set for rapid hyperlink lookup (Undirected)
    hyperlink_set = set()
    for p1, p2 in pair_list:
        hyperlink_set.add(tuple(sorted((p1, p2))))

    # Partitioning logic for parallel processing segments
    editors = list(editor_histories.keys())
    total_len = len(editors)
    part_map = {1: (0, 1/40), 2: (1/40, 5/40), 3: (5/40, 10/40), 4: (10/40, 1)}
    s_rat, e_rat = part_map.get(int(part), (0, 1))
    start_idx = int(total_len * s_rat)
    end_idx = total_len if int(part) == 4 else int(total_len * e_rat)

    # Repositories for IET distributions and unique non-hyperlink pairs
    IETs = Counter()
    pair_non_link = set()

    # Stage 1: Randomized Sampling and IET Computation
    for neditor in tqdm(range(start_idx, end_idx), desc=f"Processing Non-hyperlink Part {part}"):
        editor = editors[neditor]
        # Chronological sorting of individual history
        history = [(parse_datetime(e[0]), e[1]) for e in editor_histories[editor]]
        history.sort(key=lambda x: x[0])
        
        page_list = list({e[1] for e in history})
        if len(page_list) < 2:
            continue

        # Determine target_count: The number of existing hyperlinks to match for sampling
        target_count = 0
        for p in combinations(sorted(page_list), 2):
            if p in hyperlink_set:
                target_count += 1
        
        if target_count == 0:
            continue

        # Extract non-hyperlink pairs via randomized sampling to minimize memory overhead
        random_pairs = set()
        
        n_pages = len(page_list)
        max_possible_pairs = (n_pages * (n_pages - 1)) // 2
        
        # Ensure the target does not exceed the available pool of non-hyperlink pairs
        actual_target = min(target_count, max_possible_pairs - target_count)
        
        attempts = 0
        while len(random_pairs) < actual_target:
            attempts += 1
            if attempts > actual_target * 100: # Safety break to prevent infinite loops
                break
                
            p1, p2 = random.sample(page_list, 2)
            pair = tuple(sorted((p1, p2)))
            
            # Include only if the pair is structurally disconnected and not already sampled
            if pair not in hyperlink_set and pair not in random_pairs:
                random_pairs.add(pair)

        # Map page occurrences for recurrence interval extraction
        single_editor_history = defaultdict(list)
        for ts, pg in history:
            single_editor_history[pg].append((ts, pg))

        # Calculate intervals for the sampled non-hyperlink pairs
        for p1, p2 in random_pairs:
            # Combine and sort all revision events for the pair
            total_ts = sorted(single_editor_history[p1] + single_editor_history[p2], key=lambda x: x[0])
            
            occurred_non_link = False
            for i in range(1, len(total_ts)):
                t1, pg1 = total_ts[i-1]
                t2, pg2 = total_ts[i]
                
                # Capture intervals between transitions of different pages in the pair
                if pg1 != pg2:
                    IETs[(t2 - t1).total_seconds()] += 1
                    occurred_non_link = True
            
            if occurred_non_link:
                pair_non_link.add((p1, p2))

    # Stage 2: Results Serialization
    iet_output_file_path = os.path.join(TARGET_DIR, f'editor_longterm_IETs_{page_edit_number}_{editor_edit_number}_nonhyperlink_part{part}.pickle')
    with open(iet_output_file_path, 'wb') as fw:
        pickle.dump(dict(IETs), fw)

    pair_output_file_path = os.path.join(TARGET_DIR, f'editor_longterm_pairs_{page_edit_number}_{editor_edit_number}_nonhyperlink_part{part}.pickle')
    with open(pair_output_file_path, 'wb') as fw:
        pickle.dump(pair_non_link, fw)

    print(f"[SUCCESS] Non-hyperlink IET analysis for part {part} completed: {iet_output_file_path}")
returnturn

In [ ]:
import os
import pickle
from collections import defaultdict

def Merge_longterm_IETs_nonhyperlink(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Consolidates partitioned non-hyperlink long-term IET results and generates 
    a statistical summary for comparative structural analysis.
    """
    
    # DATA_DIR: Source directory for partitioned IET result files
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(DATA_DIR, exist_ok=True)
    
    # Initializing a repository for the consolidated IET distribution
    IETS = defaultdict(int)
    
    # Stage 1: Iterative Consolidation of Result Partitions
    for part in range(1, 5):
        # Construct the absolute file path for each partitioned non-hyperlink IET segment
        part_file_path = os.path.join(DATA_DIR, f'editor_longterm_IETs_{page_edit_number}_{editor_edit_number}_nonhyperlink_part{part}.pickle')
        
        if not os.path.exists(part_file_path):
            continue

        with open(part_file_path, 'rb') as fr:
            iets_dict = pickle.load(fr)
            
        # Update the global frequency distribution
        for iet, count in iets_dict.items():
            IETS[iet] += count
            
        # Explicit memory management
        del iets_dict

    # Stage 2: Serialize Integrated Control Dataset
    integrated_iet_file_path = os.path.join(DATA_DIR, f'editor_longterm_IETs_{page_edit_number}_{editor_edit_number}_nonhyperlink.pickle')
    with open(integrated_iet_file_path, 'wb') as fw:
        pickle.dump(dict(IETS), fw)
        
    # Stage 3: Statistical Summary and Reporting
    def get_stats_text(iet_dict, label):
        """
        Calculates descriptive statistics for the consolidated IET distribution.
        """
        if not iet_dict:
            return f"--- {label} Statistics ---\nNo Data Found.\n"
        
        # Sort by duration for percentile-based calculations
        sorted_items = sorted(iet_dict.items())
        total_transitions = sum(count for _, count in sorted_items)
        total_sum = sum(sec * count for sec, count in sorted_items)
        mean_iet = total_sum / total_transitions
        
        # Median Calculation: Identifying the 50th percentile of total transitions
        median_idx = (total_transitions + 1) / 2
        cum_count = 0
        median_iet_val = None
        for sec, count in sorted_items:
            cum_count += count
            if cum_count >= median_idx:
                median_iet_val = sec
                break
        
        return (
            f"--- {label} Statistics ---\n"
            f"Total Transitions : {total_transitions:,}\n"
            f"Mean IET          : {mean_iet:.4f} s\n"
            f"Median IET        : {median_iet_val:.4f} s\n"
            f"Min / Max IET     : {sorted_items[0][0]} / {sorted_items[-1][0]}\n"
        )

    # Generate the statistical summary text
    stats_text = get_stats_text(IETS, "Non-Hyperlink Long-term IET")
    
    # Serialize the summary report to a standardized text file
    stats_report_file_path = os.path.join(DATA_DIR, f"statistics_longterm_IETs_{page_edit_number}_{editor_edit_number}_nonhyperlink.txt")
    with open(stats_report_file_path, "w", encoding="utf-8") as f:
        f.write(stats_text)
        
    print(f"[FINISH] Non-hyperlink IET consolidation complete. Report saved to: {stats_report_file_path}")
    return

In [ ]:
import os
import pickle

def Merge_longterm_pairs_nonhyperlink(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Consolidates partitioned non-hyperlink interaction pairs into a unified dataset.
    This serves as the finalized control group for structural recurrence analysis.
    """
    
    # DATA_DIR: Source directory for partitioned non-hyperlink pair results
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(DATA_DIR, exist_ok=True)

    # Initializing a global set to aggregate unique non-structural edges
    PAIR_nonlink = set()
    
    # Stage 1: Iterative Consolidation of Result Partitions
    for part in range(1, 5):
        # Construct the absolute path for each partitioned non-hyperlink pair segment
        part_file_path = os.path.join(DATA_DIR, f'editor_longterm_pairs_{page_edit_number}_{editor_edit_number}_nonhyperlink_part{part}.pickle')
        
        if not os.path.exists(part_file_path):
            continue

        with open(part_file_path, 'rb') as fr:
            # RESULTS contains the set of unique pairs for the current partition
            partition_results = pickle.load(fr)
        
        # Merge the partition set into the global collection
        PAIR_nonlink.update(partition_results)
        
        # Explicitly release memory
        del partition_results

    # Stage 2: Data Serialization
    # Define the standardized output path for the integrated non-hyperlink pairs
    integrated_nonlink_file_path = os.path.join(DATA_DIR, f'editor_longterm_pairs_{page_edit_number}_{editor_edit_number}_nonhyperlink.pickle')
    
    with open(integrated_nonlink_file_path, 'wb') as fw:
        pickle.dump(PAIR_nonlink, fw)

    print(f"[FINISH] Non-hyperlink pair consolidation complete. Saved to: {integrated_jaccard_file_path}")
    return

### Step 2-4: Get_pdf (+ binning)

In [ ]:
import os
import pickle
import numpy as np

def Get_binning_pdf(BASE_DIR, IET_name, task, page_edit_number, editor_edit_number, lin_threshold=100, num_log_bins=150):
    """
    Generates a normalized PDF from IET distributions using a hybrid linear-logarithmic binning strategy.
    
    Parameters:
    - IET_name: 'longterm', 'overall', or 'immediate'
    - task: 'hyperlink', 'nonhyperlink', 'forward', 'backward', 'reciprocal', or ''
    """
    
    def Get_hybrid_pdf(iets_dict, lin_threshold=100, num_log_bins=150):
        """
        Core logic for hybrid bin construction and density estimation.
        """
        if not iets_dict:
            print("[WARNING] Input IET dictionary is empty.")
            return [], []
        
        # Stage 1: Data Preparation
        # Filter for positive intervals to maintain logarithmic validity
        keys = np.array(list(iets_dict.keys()))
        values = np.array(list(iets_dict.values()))

        mask = keys > 0
        keys, values = keys[mask], values[mask]

        if len(keys) == 0: 
            print("[WARNING] No positive IET keys found.")
            return [], []
        
        # Stage 2: Hybrid Bin Definition
        max_val = np.max(keys)
        if max_val <= lin_threshold:
            # Entirely linear if data range is within the threshold
            bins = np.arange(1, max_val + 2)
        else:
            # Combine linear sequence with logarithmic scale
            lin_bins = np.arange(1, lin_threshold + 1)
            log_bins = np.logspace(np.log10(lin_threshold), np.log10(max_val + 1), num_log_bins + 1)
            bins = np.unique(np.concatenate([lin_bins, log_bins]))

        # Stage 3: PDF Estimation
        # 'weights' applies frequency counts, 'density' normalizes the result by bin width
        pdf, bin_edges = np.histogram(keys, bins=bins, weights=values, density=True)

        # Calculate geometric centers for optimal logarithmic visualization
        bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])

        # Filter out zero-density bins to ensure clean log-log visualization
        nonzero = pdf > 0
        return bin_centers[nonzero], pdf[nonzero]

    # DATA_DIR: Source directory for merged IET datasets
    # TARGET_DIR: Destination for serialized PDF coordinate data
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    TARGET_DIR = os.path.join(DATA_DIR, 'IETs_pdf')
    os.makedirs(TARGET_DIR, exist_ok=True)

    # Standardized file path construction
    input_file_name = f'editor_{IET_name}_IETs_{page_edit_number}_{editor_edit_number}_{task}.pickle'
    input_file_path = os.path.join(DATA_DIR, input_file_name)
    
    with open(input_file_path, 'rb') as fr:
        IETs = pickle.load(fr)

    # Execute PDF calculation
    x_vals, y_vals = Get_hybrid_pdf(IETs, lin_threshold, num_log_bins)
    
    # Consolidate coordinates for serialization
    pdf_results = {'x_vals': x_vals, 'y_vals': y_vals}
    
    output_file_name = f'editor_{IET_name}_binning_pdf_{page_edit_number}_{editor_edit_number}_{task}.pickle'
    output_file_path = os.path.join(TARGET_DIR, output_file_name)
    
    with open(output_file_path, 'wb') as fw:
        pickle.dump(pdf_results, fw)
        
    print(f"[FINISH] Hybrid PDF serialized to: {output_file_path}")
    return

Get_binning_pdf(1000,1000, lin_threshold=100, num_log_bins=150)

In [ ]:
import os
import pickle
import numpy as np

def Get_normal_pdf(BASE_DIR, IET_name, task, page_edit_number, editor_edit_number):
    """
    Calculates the discrete probability distribution (PMF) for observed IET values.
    Provides an empirical distribution without binning-induced aggregation.
    
    Parameters:
    - IET_name: 'longterm', 'overall', or 'immediate'
    - task: 'hyperlink', 'nonhyperlink', 'forward', 'backward', 'reciprocal', 'uni', 'uni_entire'
    """
    
    def Get_pdf(iets_dict):
        """
        Calculates the normalized discrete probability for each IET entry.
        """
        if not iets_dict:
            print("[WARNING] Input IET dictionary is empty.")
            return [], []

        # Stage 1: Sort and Extract Empirical Observations
        # Sorting ensures chronological consistency along the x-axis
        sorted_items = sorted(iets_dict.items())
        keys = np.array([item[0] for item in sorted_items])
        values = np.array([item[1] for item in sorted_items])
        
        # Stage 2: Calculate Normalized Probabilities
        # Probability mass is calculated as the relative frequency of each unique interval
        total_sum = np.sum(values)
        if total_sum == 0:
            return [], []
            
        pdf = values / total_sum

        return keys, pdf

    # DATA_DIR: Source directory for merged IET datasets
    # TARGET_DIR: Destination for serialized normal PDF (PMF) data
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    TARGET_DIR = os.path.join(DATA_DIR, 'IETs_pdf')
    os.makedirs(TARGET_DIR, exist_ok=True)

    # Standardized file path construction for input data
    input_file_name = f'editor_{IET_name}_IETs_{page_edit_number}_{editor_edit_number}_{task}.pickle'
    input_file_path = os.path.join(DATA_DIR, input_file_name)
    
    try:
        with open(input_file_path, 'rb') as fr:
            IETs = pickle.load(fr)
    except FileNotFoundError:
        print(f"[ERROR] Source file not found: {input_file_path}")
        return

    # Execute discrete probability calculation
    print(f"[PROCESS] Generating Normal PDF for: {IET_name}_{task}")
    x_vals, y_vals = Get_pdf(IETs)
    
    # Consolidate results for serialization
    pdf_results = {'x_vals': x_vals, 'y_vals': y_vals}
    
    # Construct output file path following the standardized naming convention
    output_file_name = f'editor_{IET_name}_normal_pdf_{page_edit_number}_{editor_edit_number}_{task}.pickle'
    output_file_path = os.path.join(TARGET_DIR, output_file_name)
    
    with open(output_file_path, 'wb') as fw:
        pickle.dump(pdf_results, fw)
        
    print(f"[FINISH] Normal PDF serialized to: {output_file_path}")
    return

Get_binning_pdf(1000,1000, lin_threshold=100, num_log_bins=150)

## Step 2-5: Check_peak

### Step 2-5-1: Detect peak

In [ ]:
import os
import pickle
import numpy as np
from scipy.signal import find_peaks
from matplotlib import pyplot as plt

def Detect_peak(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Identifies and visualizes characteristic peaks in the IET PDF to detect 
    behavioral periodicities or system-level timing constraints.
    """
    
    # DATA_DIR: Source directory for processed PDF coordinate data
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    
    def load_files():
        """
        Retrieves the finalized IET PDF dataset for peak analysis.
        """
        # Standardized file path following the research pipeline convention
        input_file_path = os.path.join(DATA_DIR, f'Final_iets_{page_edit_number}_{editor_edit_number}.pickle')
        with open(input_file_path, 'rb') as fr:
            for_save = pickle.load(fr)
        return for_save

    # Stage 1: Data Preparation
    pdf_data = load_files()
    
    # Extracting coordinates from the target distribution (e.g., 'random' or 'hyperlink')
    # Converting to numpy arrays for signal processing compatibility
    x_vals = np.array(pdf_data['random']['x_vals'])
    y_vals = np.array(pdf_data['random']['y_vals'])

    # Stage 2: Peak Detection via Signal Processing
    # prominence: The vertical distance between the peak and its lowest contour line
    # Tuning this value is critical for distinguishing signal from noise in log-log space
    prominence_threshold = 1e-8
    peaks, properties = find_peaks(y_vals, prominence=prominence_threshold)

    # Output identified peak coordinates for quantitative verification
    print(f"=== Peak Detection Results (Prominence > {prominence_threshold}) ===")
    for idx in peaks:
        print(f"🔺 Peak identified at: x = {x_vals[idx]:.2e} sec, y = {y_vals[idx]:.2e}")

    # Stage 3: Visual Verification (Log-Log Plot)
    plt.figure(figsize=(8, 6))
    plt.plot(x_vals, y_vals, color='red', linewidth=1.5, label='PDF Distribution')
    
    # Highlight identified peaks with blue 'x' markers
    plt.plot(x_vals[peaks], y_vals[peaks], 'bx', markersize=8, label='Detected Peaks')
    
    # Applying logarithmic scales for heavy-tailed distribution visualization
    plt.xscale('log')
    plt.yscale('log')
    
    plt.xlabel('Inter-Event Time (sec)', fontsize=13)
    plt.ylabel('P(IET)', fontsize=13)
    plt.title(f'Peak Detection for {page_edit_number}/{editor_edit_number}', fontsize=14)
    
    plt.legend(frameon=True)
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.tight_layout()
    
    # Display the verification plot
    plt.show()
    
    return peaks, properties

### Step 2-5-2: Check_8minute

In [ ]:
import os
import pickle
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import ciso8601

def Check_long_term_8peak_hyperlink(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Scans the 470-500 second IET window to identify the specific editors and 
    structural page pairs contributing to the observed 8-minute peak.
    """
    
    # DATA_DIR: Source directory for history logs and interaction pairs
    # TARGET_DIR: Destination for peak-specific analytical results
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    def load_files():
        """
        Retrieves serialized interaction pairs and filtered editor histories.
        """
        pair_file_path = os.path.join(DATA_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        hist_file_path = os.path.join(DATA_DIR, f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        
        with open(pair_file_path, 'rb') as fr: pair_list = pickle.load(fr)
        with open(hist_file_path, 'rb') as fr: editor_histories = pickle.load(fr)
        return pair_list, editor_histories

    pair_list, editor_histories = load_files()
    print('[INFO] Input datasets loaded for peak-window analysis.')
    
    # Stage 1: Build Adjacency Matrix for Structural Context
    adj = defaultdict(set)
    for p1, p2 in pair_list:
        adj[p1].add(p2)
    
    parse_datetime = ciso8601.parse_datetime
    editors = list(editor_histories.keys())
    
    # Target window repository: Mapping seconds to Counters of (Editor, Pairs, Category)
    # Focus: 470s to 500s (~8 minutes)
    peak_window_dict = {i: Counter() for i in range(470, 501)}

    # Stage 2: Temporal Scanning within the 8-minute Window
    for editor in tqdm(editors, desc="Scanning 8-minute Window"):
        # Chronological sorting of individual history
        history = sorted([(parse_datetime(t_str), pg) for t_str, pg in editor_histories[editor]], key=lambda x: x[0])
        
        visited_pages = {event[1] for event in history}
        if len(visited_pages) < 2: continue

        # Map page occurrences for long-term interval extraction
        page_to_times = defaultdict(list)
        for ts, pg in history:
            page_to_times[pg].append((ts, pg))

        # Identify hyperlinked pairs in the editor's subspace
        local_links = []
        for p1 in visited_pages:
            if p1 in adj:
                for p2 in (adj[p1] & visited_pages):
                    local_links.append((p1, p2))

        processed_pairs = set()
        for p1, p2 in local_links:
            if p1 == p2: continue
            
            norm_pair = tuple(sorted((p1, p2)))
            if norm_pair in processed_pairs: continue
            
            is_reciprocal = (p2 in adj and p1 in adj[p2])
            
            # Combine all timestamps for the page pair
            combined_ts = sorted(page_to_times[p1] + page_to_times[p2], key=lambda x: x[0])
            
            for i in range(1, len(combined_ts)):
                t1, pg1 = combined_ts[i-1]
                t2, pg2 = combined_ts[i]
                
                if pg1 != pg2:
                    deltat = int((t2 - t1).total_seconds())
                    
                    # Capture transitions strictly within the anomalous peak range
                    if deltat in peak_window_dict:
                        target_counter = peak_window_dict[deltat]

                        if is_reciprocal:
                            target_counter[(editor, (p1, p2), 'reciprocal')] += 1
                        else:
                            # Link direction alignment logic
                            # forward: Path matches link orientation / backward: Path opposes link orientation
                            is_forward = (p2 in adj[p1] and pg1 == p1) or (p1 in adj[p2] and pg1 == p2)
                            category = 'forward' if is_forward else 'backward'
                            target_counter[(editor, (p1, p2), category)] += 1

            processed_pairs.add(norm_pair)

    # Stage 3: Data Serialization
    # Output file contains granular identity data for forensic behavioral analysis
    output_file_name = f'8minute_peak_data_{page_edit_number}_{editor_edit_number}.pickle'
    output_file_path = os.path.join(TARGET_DIR, output_file_name)

    with open(output_file_path, 'wb') as fw:
        pickle.dump(peak_window_dict, fw)

    print(f"[FINISH] Forensic peak data successfully serialized to: {output_file_path}")
    return

In [ ]:
import os
import pickle
import random
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import ciso8601
from itertools import combinations

def Get_8minute_iets_nonlink(BASE_DIR, page_edit_number, editor_edit_number, part):
    """
    Scans the 470-500s IET window for non-hyperlinked page pairs to establish 
    a behavioral baseline for forensic peak analysis.
    """
    # Set seed for reproducibility in stochastic pair sampling
    random.seed(20251103)
    
    # DATA_DIR: Source directory for histories and interaction pairs
    # TARGET_DIR: Destination for control group peak data
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    TARGET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    def load_files():
        """
        Retrieves serialized interaction pairs and filtered editor histories.
        """
        pair_file_path = os.path.join(DATA_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        hist_file_path = os.path.join(DATA_DIR, f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        
        with open(pair_file_path, 'rb') as fr: pair_list = pickle.load(fr)
        with open(hist_file_path, 'rb') as fr: editor_histories = pickle.load(fr)
        return pair_list, editor_histories

    pair_list, editor_histories = load_files()
    print('[INFO] Input files loaded for non-hyperlink peak analysis.')
    
    parse_datetime = ciso8601.parse_datetime
    
    # Target window repository for forensic tracking (470s to 500s)
    peak_window_dict = {i: Counter() for i in range(470, 501)}
    
    # Construct a set for rapid hyperlink lookup (Undirected)
    hyperlink_set = set()
    for p1, p2 in pair_list:
        hyperlink_set.add(tuple(sorted((p1, p2))))

    # Partitioning logic for parallel segment processing
    editors = list(editor_histories.keys())
    total_len = len(editors)
    part_map = {1: (0, 1/40), 2: (1/40, 5/40), 3: (5/40, 10/40), 4: (10/40, 1)}
    s_rat, e_rat = part_map.get(int(part), (0, 1))
    start_idx = int(total_len * s_rat)
    end_idx = total_len if int(part) == 4 else int(total_len * e_rat)

    # Stage 1: Randomized Sampling and Forensic Scanning
    for neditor in tqdm(range(start_idx, end_idx), desc=f"Scanning Non-link Part {part}"):
        editor = editors[neditor]
        # Chronological sorting of editor history
        history = sorted([(parse_datetime(e[0]), e[1]) for e in editor_histories[editor]], key=lambda x: x[0])
        
        page_list = list({e[1] for e in history})
        if len(page_list) < 2: continue

        # Identify target count (matching the number of structural links for the editor)
        target_count = 0
        for p in combinations(sorted(page_list), 2):
            if p in hyperlink_set:
                target_count += 1
        
        if target_count == 0: continue

        # Stochastic extraction of non-hyperlink pairs
        random_pairs = set()
        n_pages = len(page_list)
        max_possible_pairs = (n_pages * (n_pages - 1)) // 2
        actual_target = min(target_count, max_possible_pairs - target_count)
        
        attempts = 0
        while len(random_pairs) < actual_target:
            attempts += 1
            if attempts > actual_target * 100: break # Safety break
                
            p1, p2 = random.sample(page_list, 2)
            pair = tuple(sorted((p1, p2)))
            
            if pair not in hyperlink_set and pair not in random_pairs:
                random_pairs.add(pair)

        # Map page occurrences for interval extraction
        single_editor_history = defaultdict(list)
        for ts, pg in history:
            single_editor_history[pg].append((ts, pg))

        # Stage 2: Interval Extraction for Peak Identification
        for p1, p2 in random_pairs:
            # Combine and sort timestamps for the sampled pair
            total_ts = sorted(single_editor_history[p1] + single_editor_history[p2], key=lambda x: x[0])
            
            for i in range(1, len(total_ts)):
                t1, pg1 = total_ts[i-1]
                t2, pg2 = total_ts[i]
                
                if pg1 != pg2:
                    deltat = int((t2 - t1).total_seconds())
                    
                    # Capture transitions within the specific peak window
                    if deltat in peak_window_dict:
                        target_counter = peak_window_dict[deltat]
                        # Label as non-hyperlinked for structural comparison
                        target_counter[(editor, (p1, p2), 'non-hyperlinked')] += 1

    # Stage 3: Data Serialization
    output_file_name = f'8minute_peak_data_nonhyperlink_part{part}_{page_edit_number}_{editor_edit_number}.pickle'
    output_file_path = os.path.join(TARGET_DIR, output_file_name)
    
    with open(output_file_path, 'wb') as fw:
        pickle.dump(peak_window_dict, fw)

    print(f"[FINISH] Non-hyperlink forensic data serialized to: {output_file_path}")
    return

## Step 2-6: IET with editor type

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

def Classify_editors(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Classifies editors into Specialists, Generalists, and Bots based on 
    structural diversity and behavioral alignment metrics.
    """
    
    # DATA_DIR: Source directory for diversity and alignment datasets
    # TARGET_DIR: Destination for serialized editor classification results
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset')
    IET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(IET_DIR, exist_ok=True)

    def load_files():
        """
        Retrieves serialized Jaccard indices, diversity features, and bot metadata.
        """
        # Load Undirected Jaccard Indices (Structural Alignment)
        jaccard_file_path = os.path.join(DATA_DIR, 'Jaccard', f'Jaccard_{page_edit_number}_{editor_edit_number}.pickle')
        with open(jaccard_file_path, 'rb') as fr:
            j_dict_raw = pickle.load(fr)
        j_dict_undir = j_dict_raw['undirection']

        # Load Diversity Metrics (Entropy and Inversed Simpson Index)
        # Note: 'Entropy_IHHI' in legacy code updated to 'Entropy_inverse_lambda' for consistency
        diversity_file_path = os.path.join(DATA_DIR, f'Entropy_inverse_lambda_{page_edit_number}_{editor_edit_number}.pickle')
        with open(diversity_file_path, 'rb') as fr:
            diversity_dict = pickle.load(fr)

        # Load Bot Identifiers
        bot_file_path = os.path.join(DATA_DIR, 'Bot_list_with_info_id.pickle')
        with open(bot_file_path, 'rb') as fr:
            bot_list_raw = pickle.load(fr)
            
        bot_lookup = {f"{b['id']}_{b['name']}": b['name'] for b in bot_list_raw}
        
        return j_dict_undir, diversity_dict, bot_lookup

    # Stage 1: Data Acquisition
    J_dict, Entropy_dict, Bot_ID = load_files()
    ln2 = np.log(2)

    # Classification Repositories
    Specialist_J, Generalist_J, Bot_J = [], [], []
    Specialist_ID, Generalist_ID = [], []
    exception_count = 0

    # Stage 2: Categorization Logic
    for editor, jaccard in J_dict.items():
        if jaccard == 'NaN':
            continue
            
        # Extract features: [Entropy, Lambda, Inversed Simpson Index, ...]
        features = Entropy_dict.get(editor)
        if not features:
            continue
            
        entropy_val = features[0]
        inv_simpson_val = features[2]
        is_bot = editor in Bot_ID

        if is_bot:
            Bot_J.append(jaccard)
        else:
            # Human Editor Classification
            if inv_simpson_val <= 2 and entropy_val <= ln2:
                # Specialist: Low diversity, focused activity
                Specialist_J.append(jaccard)
                Specialist_ID.append(editor)
            elif inv_simpson_val > 2 and entropy_val > ln2:
                # Generalist: High diversity, distributed activity
                Generalist_J.append(jaccard)
                Generalist_ID.append(editor)
            else:
                # Ambiguous cases not strictly meeting diversity thresholds
                exception_count += 1

    # Stage 3: Results Serialization
    editor_classification = {
        'Generalist': Generalist_ID, 
        'Specialist': Specialist_ID, 
        'Bot': list(Bot_ID.keys())
    }

    output_file_path = os.path.join(IET_DIR, f'editor_type_{page_edit_number}_{editor_edit_number}.pickle')
    with open(output_file_path, 'wb') as fw:
        pickle.dump(editor_classification, fw)

    print(f"=== Editor Classification Summary ===")
    print(f"Specialists Identified : {len(Specialist_ID):,}")
    print(f"Generalists Identified : {len(Generalist_ID):,}")
    print(f"Bots Registered        : {len(Bot_ID):,}")
    print(f"Exceptions Handled     : {exception_count:,}")
    print(f"[FINISH] Classification serialized to: {output_file_path}")
    
    return editor_classification

In [ ]:
import os
import pickle
import math
import random
import ciso8601
from collections import Counter, defaultdict
from itertools import combinations
from tqdm.auto import tqdm

def Get_type_editor_longterm_IET(BASE_DIR, page_edit_number, editor_edit_number, part):
    """
    Analyzes long-term IET distributions categorized by editor types (Specialist, Generalist, Bot).
    Aggregates recurrence data across both hyperlink and non-hyperlink subspaces.
    """
    
    # DATA_DIR: Source directory for histories and classification metadata
    # TARGET_DIR: Destination for serialized category-wise IET distributions
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    IET_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(IET_DIR, exist_ok=True)
    
    def load_files():
        """
        Retrieves interaction pairs, editor histories, and pre-defined editor classifications.
        """
        pair_file_path = os.path.join(DATA_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        hist_file_path = os.path.join(DATA_DIR, f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        type_file_path = os.path.join(IET_DIR, f'editor_type_{page_edit_number}_{editor_edit_number}.pickle')

        with open(pair_file_path, 'rb') as fr: pair_list = pickle.load(fr)
        with open(hist_file_path, 'rb') as fr: editor_histories = pickle.load(fr)
        with open(type_file_path, 'rb') as fr: editor_type = pickle.load(fr)
        return pair_list, editor_histories, editor_type

    # Stage 1: Initialization and Metadata Loading
    pair_list, editor_histories, editor_type = load_files()
    print('[INFO] Metadata and history logs loaded successfully.')
    
    Specialist_IDs = set(editor_type['Specialist'])
    Generalist_IDs = set(editor_type['Generalist'])
    Bot_IDs = set(editor_type['Bot'])
    
    # Construct adjacency lookup for structural connectivity verification
    adj = defaultdict(set)
    hyperlink_set = set()
    for p1, p2 in pair_list:
        adj[p1].add(p2)
        hyperlink_set.add(tuple(sorted((p1, p2))))
        
    del pair_list # Memory optimization
        
    parse_datetime = ciso8601.parse_datetime
    editors = list(editor_histories.keys())
    total_length = len(editors)
    
    # Partitioning logic for distributed processing
    num_parts = 40
    start_index = math.floor(total_length * (0 if part == 1 else (1/40 if part == 2 else (5/40 if part == 3 else 10/40))))
    end_index = total_length if part == 4 else math.floor(total_length * (1/40 if part == 1 else (5/40 if part == 2 else 10/40)))
    part_range = [start_index, end_index]

    # Repositories for Group-wise Distributions
    S_lt_IETs, G_lt_IETs, B_lt_IETs = Counter(), Counter(), Counter()
    S_pair_link, G_pair_link, B_pair_link = set(), set(), set()
    Individual_stats = {'Generalist': {}, 'Specialist': {}, 'Bot': {}}

    def get_median_from_counter(counts):
        """Calculates the median value from a frequency Counter."""
        sorted_items = sorted(counts.items())
        total_count = sum(counts.values())
        mid = total_count / 2
        cumulative_count = 0
        for val, count in sorted_items:
            cumulative_count += count
            if cumulative_count >= mid: return val
        return None

    def lt_recurrence_check(global_IETs, global_pairs, page_list, history, hyperlink_set, adj):
        """
        Extracts long-term IETs for an individual editor, considering both 
        structural links and a sampled control group of non-links.
        """
        local_IETs = Counter()
        processed_pairs = set()
        page_to_times = defaultdict(list)
        for ts, pg in history:
            page_to_times[pg].append((ts, pg))

        # A. Target Sampling for Control Group
        target_count = sum(1 for p in combinations(sorted(page_list), 2) if p in hyperlink_set)
        n_pages = len(page_list)
        max_possible = (n_pages * (n_pages - 1)) // 2
        actual_target = min(target_count, max_possible - target_count)

        random_pairs = set()
        if actual_target > 0:
            random.seed(20251103)
            attempts = 0
            while len(random_pairs) < actual_target:
                attempts += 1
                if attempts > actual_target * 100: break
                p1, p2 = random.sample(page_list, 2)
                pair = tuple(sorted((p1, p2)))
                if pair not in hyperlink_set and pair not in random_pairs:
                    random_pairs.add(pair)

        # B. IET Extraction: Sampled Non-hyperlinks and Existing Hyperlinks
        # Merging both subspaces to evaluate overall group-wise recurrence
        relevant_pairs = random_pairs | {tuple(sorted((p1, p2))) for p1 in page_list if p1 in adj for p2 in (adj[p1] & set(page_list))}

        for p1, p2 in relevant_pairs:
            if p1 == p2 or tuple(sorted((p1, p2))) in processed_pairs: continue
            
            combined_ts = sorted(page_to_times[p1] + page_to_times[p2], key=lambda x: x[0])
            occurred = False
            for i in range(1, len(combined_ts)):
                t1, pg1 = combined_ts[i-1]
                t2, pg2 = combined_ts[i]
                if pg1 != pg2:
                    dt = (t2 - t1).total_seconds()
                    global_IETs[dt] += 1
                    local_IETs[dt] += 1
                    occurred = True
            
            processed_pairs.add(tuple(sorted((p1, p2))))
            if occurred: global_pairs.add(tuple(sorted((p1, p2))))

        total_local = sum(local_IETs.values())
        if total_local == 0: return global_IETs, global_pairs, None, None

        mean_val = sum(v * c for v, c in local_IETs.items()) / total_local
        median_val = get_median_from_counter(local_IETs)
        return global_IETs, global_pairs, mean_val, median_val

    # Stage 2: Segmented Iteration and Categorical Aggregation
    for neditor in tqdm(range(part_range[0], part_range[1]), desc=f"Part {part}"):
        editor = editors[neditor]
        history = sorted([(parse_datetime(t_str), pg) for t_str, pg in editor_histories[editor]], key=lambda x: x[0])
        page_list = list({e[1] for e in history})
        if len(page_list) < 2: continue
            
        # Dispatch to the appropriate category based on classification metadata
        if editor in Generalist_IDs:
            G_lt_IETs, G_pair_link, mean, median = lt_recurrence_check(G_lt_IETs, G_pair_link, page_list, history, hyperlink_set, adj)
            Individual_stats['Generalist'][editor] = {'Mean': mean, 'Median': median}
        elif editor in Specialist_IDs:
            S_lt_IETs, S_pair_link, mean, median = lt_recurrence_check(S_lt_IETs, S_pair_link, page_list, history, hyperlink_set, adj)
            Individual_stats['Specialist'][editor] = {'Mean': mean, 'Median': median}
        elif editor in Bot_IDs:
            B_lt_IETs, B_pair_link, mean, median = lt_recurrence_check(B_lt_IETs, B_pair_link, page_list, history, hyperlink_set, adj)
            Individual_stats['Bot'][editor] = {'Mean': mean, 'Median': median}

    # Stage 3: Data Serialization
    iet_results = {
        'Specialist_longterm_IETs': dict(S_lt_IETs), 
        'Generalist_longterm_IETs': dict(G_lt_IETs), 
        'Bot_longterm_IETs': dict(B_lt_IETs), 
        'Individual': Individual_stats
    }
    
    iet_file_path = os.path.join(IET_DIR, f'editor_type_longterm_IETs_{page_edit_number}_{editor_edit_number}_part{part}.pickle')
    with open(iet_file_path, 'wb') as fw:
        pickle.dump(iet_results, fw)
                            
    pair_results = {
        'Specialist': S_pair_link, 'Generalist': G_pair_link, 'Bot': B_pair_link
    }
    pair_file_path = os.path.join(IET_DIR, f'editor_type_longterm_pairs_{page_edit_number}_{editor_edit_number}_part{part}.pickle')
    with open(pair_file_path, 'wb') as fw:
        pickle.dump(pair_results, fw)

    print(f"[SUCCESS] Category-wise long-term IET analysis complete for Part {part}.")
    return

In [ ]:
import os
import pickle
from collections import defaultdict

def Merge_type_longterm_IETs(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Consolidates partitioned long-term IET results categorized by editor types.
    Merges both global frequency counters and individual-level summary statistics.
    """
    
    # DATA_DIR: Source directory for partitioned category-wise IET files
    DATA_DIR = os.path.join(BASE_DIR, 'Inter_event_time')
    os.makedirs(DATA_DIR, exist_ok=True)
    
    # Repositories for consolidated group-wide distributions
    G_lt_IETs = defaultdict(int)  # Generalist global Counter
    S_lt_IETs = defaultdict(int)  # Specialist global Counter
    B_lt_IETs = defaultdict(int)  # Bot global Counter
    
    # Repository for individual-level statistical metrics (Mean/Median)
    # Structured as { 'Category': { 'Editor_ID': {'Mean': val, 'Median': val}, ... } }
    I_lt_IETs = {'Generalist': {}, 'Specialist': {}, 'Bot': {}}
    
    # Stage 1: Iterative Consolidation of Result Partitions
    for part in range(1, 5):
        # Construct the absolute path for each partitioned file segment
        part_file_path = os.path.join(DATA_DIR, f'editor_type_longterm_IETs_{page_edit_number}_{editor_edit_number}_part{part}.pickle')
        
        if not os.path.exists(part_file_path):
            continue

        with open(part_file_path, 'rb') as fr:
            RESULTS = pickle.load(fr)
            
        # A. Process Generalist Data
        g_iets = RESULTS['Generalist_longterm_IETs']
        for iet, count in g_iets.items():
            G_lt_IETs[iet] += count
        I_lt_IETs['Generalist'].update(RESULTS['Individual']['Generalist'])
        
        # B. Process Specialist Data
        s_iets = RESULTS['Specialist_longterm_IETs']
        for iet, count in s_iets.items():
            S_lt_IETs[iet] += count
        I_lt_IETs['Specialist'].update(RESULTS['Individual']['Specialist'])
        
        # C. Process Bot Data
        b_iets = RESULTS['Bot_longterm_IETs']
        for iet, count in b_iets.items():
            B_lt_IETs[iet] += count
        I_lt_IETs['Bot'].update(RESULTS['Individual']['Bot'])
        
        # Explicitly release the partition data to optimize memory usage
        del RESULTS

    # Stage 2: Final Data Consolidation and Serialization
    # Convert defaultdicts to regular dicts for clean serialization
    Final_integrated_data = {
        'Specialist': dict(S_lt_IETs), 
        'Generalist': dict(G_lt_IETs), 
        'Bot': dict(B_lt_IETs), 
        'Individual': I_lt_IETs
    }
    
    # Define the standardized output path for the finalized integrated dataset
    integrated_file_path = os.path.join(DATA_DIR, f'editor_type_longterm_IETs_{page_edit_number}_{editor_edit_number}.pickle')
    
    with open(integrated_file_path, 'wb') as fw:
        pickle.dump(Final_integrated_data, fw)
        
    print(f"=== Category Merge Complete ===")
    print(f"Specialists : {len(I_lt_IETs['Specialist']):,}")
    print(f"Generalists : {len(I_lt_IETs['Generalist']):,}")
    print(f"Bots        : {len(I_lt_IETs['Bot']):,}")
    print(f"[FINISH] Integrated dataset saved to: {integrated_file_path}")
    
    return

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from matplotlib import font_manager as fm

# --- Stage 1: Data Acquisition and Extraction ---
def extract_mean_metrics(BASE_DIR, page_edit_number, editor_edit_number):
    """
    Loads integrated editor-type IET data and extracts mean IET lists for each category.
    """
    file_path = os.path.join(BASE_DIR, 'Inter_event_time', f'editor_type_longterm_IETs_{page_edit_number}_{editor_edit_number}.pickle')
    
    with open(file_path, 'rb') as fr:
        final_data = pickle.load(fr)

    # Dictionary to store extracted mean values
    means_repo = {
        'Bot': [],
        'Generalist': [],
        'Specialist': []
    }

    # Extract Mean values while filtering out None entries
    for category in means_repo.keys():
        category_data = final_data['Individual'][category]
        for editor in category_data.keys():
            mean_val = category_data[editor].get('Mean')
            if mean_val is not None:
                means_repo[category].append(mean_val)
                
    return means_repo

# Stage 2: Plotting and Stylistic Configuration
means_data = extract_mean_metrics(BASE_DIR, page_edit_number, editor_edit_number)

# Font configuration for academic presentation
font_path = os.path.join(BASE_DIR, 'NotoSans-VariableFont_wdth,wght.ttf')
prop = fm.FontProperties(fname=font_path)

label_fontsize = 20
tick_fontsize = 12.5

fig, ax = plt.subplots(figsize=(10, 7))

# Mapping categories to visual styles
plot_configs = {
    'Bot': (means_data['Bot'], 'green', 's'), 
    'Generalist': (means_data['Generalist'], 'lightcoral', 'o'), 
    'Specialist': (means_data['Specialist'], 'skyblue', '^')
}

# Stage 3: PDF Generation and Visualization
for name, (vals, color, marker) in plot_configs.items():
    # Calculate density histogram for probability density estimation
    y_pdf, x_edges = np.histogram(vals, bins=20, density=True)
    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    
    # Handle zero values for logarithmic scale compatibility
    y_pdf = y_pdf.astype(float)
    y_pdf[y_pdf == 0] = np.nan

    # Plot the distribution
    ax.plot(x_centers, y_pdf, marker=marker, linestyle='-', 
            label=name, c=color, markersize=12, linewidth=2, 
            markeredgecolor='white', markeredgewidth=1)

# Axis formatting: Scientific notation for x-axis (Inter-event seconds)
formatter = ScalarFormatter(useMathText=True) 
formatter.set_scientific(True)
formatter.set_powerlimits((-3, 3)) 
ax.xaxis.set_major_formatter(formatter)

# Set axis labels with TeX formatting
ax.set_xlabel(r'$\langle \tau_{\rm long} \rangle$ (sec)', fontproperties=prop, fontsize=label_fontsize)
ax.set_ylabel(r'$P(\langle \tau_{\rm long} \rangle)$', fontproperties=prop, fontsize=label_fontsize)

# Refine tick and offset text appearance
plt.draw() 
ax.xaxis.get_offset_text().set_fontproperties(prop)
ax.xaxis.get_offset_text().set_fontsize(tick_fontsize)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(prop)
    label.set_fontsize(tick_fontsize)

# Legend configuration
leg = ax.legend(prop=prop, handlelength=3.0, labelspacing=0.5, borderpad=1, frameon=False)
for line in leg.get_lines():
    line.set_linewidth(2.5) 
plt.setp(leg.get_texts(), fontsize=tick_fontsize, va='center_baseline')

# Final layout adjustments
ax.set_yscale('log')
plt.tight_layout()

# Export the figure
save_path = os.path.join(BASE_DIR, 'Results', f'editortype_IET_{page_edit_number}_{editor_edit_number}.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()

print(f"[FINISH] Categorical IET distribution plot saved to: {save_path}")

## ETC.

In [ ]:
import os
import pickle
from tqdm.auto import tqdm
from collections import defaultdict

def Get_Computation_Ratio_Final_Report(page_edit_number, editor_edit_number):
    """
    Generates a comparative report on computational complexity between 
    1:1 sampling and exhaustive non-hyperlink enumeration.
    """
    BASE_DIR = os.getcwd()
    # DATA_DIR: Source directory for history logs and validated interaction pairs
    DATA_DIR = os.path.join(BASE_DIR, 'Dataset', 'conditioned_history')
    
    def load_files():
        """
        Retrieves serialized interaction pairs and filtered editor histories.
        """
        pair_file_path = os.path.join(DATA_DIR, f'condi_pairs_{page_edit_number}_{editor_edit_number}.pickle')
        hist_file_path = os.path.join(DATA_DIR, f'condi_pages_{page_edit_number}_editors_{editor_edit_number}.pickle')
        
        with open(pair_file_path, 'rb') as fr: pair_list = pickle.load(fr)
        with open(hist_file_path, 'rb') as fr: editor_histories = pickle.load(fr)
        return pair_list, editor_histories

    # Stage 1: Network Adjacency Reconstruction
    print("[INFO] Loading datasets and reconstructing adjacency subspace...")
    pair_list, editor_histories = load_files()

    # Optimized Adjacency List for set intersection operations
    adj = defaultdict(set)
    for p1, p2 in pair_list:
        adj[p1].add(p2)
        adj[p2].add(p1)

    # Accumulators for Complexity Metrics
    total_sampling_ops = 0  # Current approach (1:1 with Hyperlinks)
    total_full_ops = 0      # Exhaustive approach (All Non-hyperlinks)
    heavy_editors_count = 0 # Count of editors with N >= 100 pages
    
    # Stage 2: Iterative Complexity Mapping
    editors = list(editor_histories.keys())
    print(f"[PROCESS] Analyzing {len(editors):,} editors for computational load...")

    for editor, history in tqdm(editor_histories.items(), desc="Quantifying Ops"):
        # Identify unique pages to define the node subspace (N)
        pages_set = {e[1] for e in history}
        n = len(pages_set)
        
        if n < 2:
            continue

        # Calculate actual structural hyperlinks (target_count) within the editor's subspace
        # Using set intersection for O(min(len)) average complexity
        target_count = 0
        for p in pages_set:
            if p in adj:
                target_count += len(adj[p] & pages_set)
        
        target_count //= 2  # Adjust for bidirectional counting in the undirected adj list
        
        if target_count == 0:
            continue

        # Cumulative Operation Count
        max_possible_pairs = n * (n - 1) // 2
        
        # Current: Balanced 1:1 Matching
        total_sampling_ops += target_count
        
        # Exhaustive: Combinatorial Expansion (Total Combinations - Hyperlinks)
        total_full_ops += (max_possible_pairs - target_count)
        
        # Monitoring Heavy Users (Drivers of N^2 complexity)
        if n >= 100:
            heavy_editors_count += 1

    # Stage 3: Statistical Reporting and Conclusion
    print("\n" + "="*50)
    print(f"📊 [COMPUTATIONAL COMPLEXITY REPORT]")
    print(f"- Total Analyzed Editors: {len(editor_histories):,}")
    print(f"- High-Activity Editors (N>=100): {heavy_editors_count:,}")
    print("-" * 50)
    print(f"1️⃣ Current Sampling Workload : {total_sampling_ops:,} pairs")
    print(f"2️⃣ Exhaustive Search Workload: {total_full_ops:,} pairs")
    print("-" * 50)
    
    if total_sampling_ops > 0:
        ratio = total_full_ops / total_sampling_ops
        print(f"📈 Conclusion: Exhaustive search increases complexity by approx. 【 {ratio:.2f}x 】")
        
        print("\n[Methodological Justification]")
        if ratio > 100:
            print(f"💡 'Full enumeration is computationally prohibitive, increasing the load by {int(ratio)}x.'")
            print(f"💡 'The N^2 expansion for {heavy_editors_count} heavy users exceeds feasible processing limits.'")
        elif ratio > 10:
            print(f"💡 'Sampling is necessary as full enumeration increases processing time by over {int(ratio)} times.'")
    else:
        print("[ERROR] No valid hyperlink pairs found in the current dataset.")
    print("="*50)

# Execution for standard thresholds
Get_Computation_Ratio_Final_Report(1000, 1000)